# 01 · Dataset exploration

Documents in the knowledge base: publication and indexing state, languages, categories, source types, versions, page metadata, missing metadata and data-quality findings.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
stats = json.loads((A.SNAP / "dataset_statistics.json").read_text())
manifest = json.loads((A.SNAP / "corpus_manifest.json").read_text())
docs, versions, chunks = stats["documents"], stats["versions"], stats["chunks"]
print("statistics generated at:", stats["generated_at"], "(read-only query: research/sql/dataset_statistics.sql)")
print("public corpus snapshot:", manifest["snapshot_sha256_16"], "-", len(manifest["items"]), "items")

## Overview

*Active* = the current approved version of a non-canceled, non-archived document of an active source - exactly what the retrieval view `searchable_chunks` exposes. Everything else is historical.

In [ ]:
overview = pd.DataFrame([
    ("Knowledge documents (all)", docs["total"]),
    ("Documents with an approved current version", docs["with_an_approved_current_version"]),
    ("Active (searchable) documents", docs["active_searchable"]),
    ("Active documents fully indexed (every chunk embedded)", docs["indexed_active (all chunks embedded)"]),
    ("Active documents without chunks", docs["active_without_chunks"]),
    ("Canceled documents", docs["canceled"]), ("Archived documents", docs["archived"]),
    ("Document versions (all)", versions["total"]),
    ("Extracted pages (documents with page metadata only)", stats["pages"]["total_pages_from_extraction_metadata"]),
    ("Documents with page metadata", stats["pages"]["documents_with_page_metadata"]),
], columns=["measure", "value"])
overview

In [ ]:
def as_table(d, name):
    return pd.Series(d, name=name).sort_values(ascending=False).rename_axis("value").reset_index(name="documents")

lang = as_table(docs["by_language"], "language"); cat = as_table(docs["by_category"], "category")
dtype = as_table(docs["by_document_type"], "type"); src = as_table(docs["by_source_type"], "source")
display(lang); display(cat); display(dtype); display(src)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [2.2, 1]})
c = cat.sort_values("documents")
axes[0].barh(c["value"], c["documents"], color=A.SINGLE, height=0.6, zorder=2)
for y, v in enumerate(c["documents"]):
    axes[0].text(v + 0.08, y, str(v), va="center", fontsize=9, color=A.INK)
axes[0].set_title(f"Active documents by category (n = {docs['active_searchable']})"); axes[0].set_xlabel("documents")
axes[0].grid(axis="x"); axes[0].grid(axis="y", visible=False)
l = lang.copy(); l["label"] = l["value"].map(lambda x: A.LANG_NAMES.get(x, x))
axes[1].bar(l["label"], l["documents"], color=A.SINGLE, width=0.55, zorder=2)
for x, v in enumerate(l["documents"]):
    axes[1].text(x, v + 0.2, str(v), ha="center", fontsize=9, color=A.INK)
axes[1].set_title("Original document language"); axes[1].set_ylabel("documents")
fig.tight_layout(); print(A.save(fig, "01_documents_by_category_and_language"))

## Versions, indexing state and jobs

In [ ]:
display(pd.Series(versions["by_status"], name="versions").rename_axis("verification status").reset_index())
display(pd.Series(versions["by_indexing_status"], name="versions").rename_axis("indexing_status flag").reset_index())
display(pd.Series(stats["announcements"], name="announcements").rename_axis("publication status").reset_index())
display(pd.Series(stats["jobs"], name="jobs").rename_axis("job type:status").reset_index())

## Missing metadata and data quality

In [ ]:
items = pd.DataFrame(manifest["items"])
quality = pd.DataFrame([
    ("Active documents with a validity end date (valid_until)", docs["with_valid_until"], docs["active_searchable"]),
    ("Public items with an official source URL", int(items["has_source_url"].sum()), len(items)),
    ("Active documents with page metadata", stats["pages"]["documents_with_page_metadata"], docs["active_searchable"]),
    ("Versions whose indexing_status flag is not 'indexed'",
     sum(v for k, v in versions["by_indexing_status"].items() if k != "indexed"), versions["total"]),
    ("Failed ingestion jobs", sum(v for k, v in stats["jobs"].items() if k.endswith(":failed")), sum(stats["jobs"].values())),
    ("Active chunks shorter than 250 characters", chunks["size_histogram_chars_active"].get("0-249", 0), chunks["active"]),
], columns=["check", "count", "out_of"])
quality["share"] = (quality["count"] / quality["out_of"]).round(3)
quality

In [ ]:
flag = sum(v for k, v in versions["by_indexing_status"].items() if k != "indexed")
display(Markdown(f"""**Interpretation.** All {docs['active_searchable']} active documents are fully indexed
({chunks['active_with_embedding']}/{chunks['active']} active chunks carry an embedding), yet {flag} version rows still carry
an `indexing_status` other than `indexed` - a bookkeeping inconsistency, not missing embeddings. Only
{docs['with_valid_until']} of {docs['active_searchable']} active documents carry a validity end date, so time-bound content
(e.g. a past course-registration period) cannot expire automatically; {int((~items['has_source_url']).sum())} of
{len(items)} public items have no official source URL. Page counts exist only for the
{stats['pages']['documents_with_page_metadata']} PDF documents and are not imputed for the others."""))

## Snapshot comparison (corpus_v1 → corpus_v2)

Both snapshots are read-only exports of the production database (`data/snapshots/<snapshot>/`). This section compares them whatever snapshot the notebook runs on.

In [ ]:
snaps = {k: json.loads((A.DATA / "snapshots" / k / "dataset_statistics.json").read_text())
         for k in ("corpus_v1", "corpus_v2") if (A.DATA / "snapshots" / k / "dataset_statistics.json").exists()}
if len(snaps) < 2:
    display(Markdown("Not evaluated: only one snapshot is available."))
else:
    a, b = snaps["corpus_v1"], snaps["corpus_v2"]
    def g(s, *path):
        for p in path: s = s.get(p, {}) if isinstance(s, dict) else {}
        return s
    comp = pd.DataFrame([(name, g(a, *p), g(b, *p)) for name, p in (
        ("snapshot time (UTC)", ("generated_at",)), ("documents (all)", ("documents", "total")),
        ("active documents", ("documents", "active_searchable")), ("fully indexed active documents", ("documents", "indexed_active (all chunks embedded)")),
        ("known pages (PDF metadata)", ("pages", "total_pages_from_extraction_metadata")), ("active chunks", ("chunks", "active")),
        ("active chunks with embedding", ("chunks", "active_with_embedding")), ("historical chunks", ("chunks", "historical")),
        ("median chunk size (chars)", ("chunks", "size_chars_active", "median")), ("mean chunk size (chars)", ("chunks", "size_chars_active", "mean")),
        ("documents with valid_until", ("documents", "with_valid_until")))], columns=["measure", "corpus_v1", "corpus_v2"])
    display(comp)
    lang = pd.DataFrame({"documents v1": g(a, "documents", "by_language"), "documents v2": g(b, "documents", "by_language"),
                         "chunks v1": g(a, "chunks", "by_language_active"), "chunks v2": g(b, "chunks", "by_language_active")}).fillna(0).astype(int)
    cat = pd.DataFrame({"documents v1": g(a, "documents", "by_category"), "documents v2": g(b, "documents", "by_category"),
                        "chunks v1": g(a, "chunks", "by_category_active"), "chunks v2": g(b, "chunks", "by_category_active")}).fillna(0).astype(int)
    display(lang); display(cat)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for ax, (tab, title) in zip(axes, ((lang, "language"), (cat, "category"))):
        y = np.arange(len(tab)); h = 0.38
        ax.barh(y - h / 2, tab["chunks v1"], h, color=A.COLORS["B0"], label="corpus_v1", zorder=2)
        ax.barh(y + h / 2, tab["chunks v2"], h, color=A.COLORS["E1"], label="corpus_v2", zorder=2)
        for yy, v1, v2 in zip(y, tab["chunks v1"], tab["chunks v2"]):
            ax.text(v1 + 3, yy - h / 2, str(v1), va="center", fontsize=8); ax.text(v2 + 3, yy + h / 2, str(v2), va="center", fontsize=8)
        ax.set_yticks(y, [A.LANG_NAMES.get(i, i) for i in tab.index]); ax.invert_yaxis()
        ax.set_title(f"Active chunks by {title}"); ax.set_xlabel("chunks"); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    axes[1].legend(loc="lower right"); fig.tight_layout(); print(A.save(fig, "01_snapshot_comparison"))

In [ ]:
inv_f = A.DATA / "snapshots" / "corpus_v2" / "document_inventory.csv"
if len(snaps) == 2 and inv_f.exists():
    inv = pd.read_csv(inv_f)
    v1_ids = {d["id"] for d in snaps["corpus_v1"]["per_document"]}
    added = inv[~inv.document_id.isin(v1_ids) & ~inv.is_canceled]
    cols = ["document_id", "title", "source_type", "language", "category", "current_status", "current_version_number",
            "pages_metadata", "current_chunks", "current_embedded", "searchable_chunks", "indexing_status", "source_url",
            "same_hash_other_documents", "same_title_other_documents", "failed_jobs_for_source"]
    display(Markdown(f"**Documents added since corpus_v1** ({len(added)}):")); display(added[cols])
    coverage = inv[~inv.is_canceled & inv.current_status.eq("approved")].assign(
        fully_embedded=lambda d: d.current_embedded == d.current_chunks, searchable=lambda d: d.searchable_chunks > 0)
    display(Markdown("**Indexing coverage (approved, non-canceled documents):**"))
    display(coverage[["fully_embedded", "searchable"]].agg(["sum", "count"]).T.assign(problem=lambda d: d["count"] - d["sum"]))
    ch2 = pd.read_csv(A.DATA / "snapshots" / "corpus_v2" / "chunk_metrics.csv")
    new_ch = ch2[ch2.active & ch2.document_id.isin(added.document_id)]
    display(Markdown(f"""**Change summary.** {len(added)} documents added ({', '.join(added.title)}); active chunks
{snaps['corpus_v1']['chunks']['active']} → {snaps['corpus_v2']['chunks']['active']}; Korean chunks
{g(snaps['corpus_v1'], 'chunks', 'by_language_active').get('ko', 0)} → {g(snaps['corpus_v2'], 'chunks', 'by_language_active').get('ko', 0)}.
The new chunks are short (median {new_ch.chars.median():.0f} characters, {int((new_ch.chars < 100).sum())} of {len(new_ch)} under
100): the PDF's layout fragments the chunker's segments - an indexing-quality issue to fix before relying on this
document for retrieval experiments. Benchmark labels predate these documents and must be reviewed for corpus_v2."""))